# Analizador de sentimiento con una RNN (LSTM)

Este notebook muestra un ejemplo básico de cómo usar una red neuronal recurrente para clasificar frases como positivas o negativas.

In [29]:
# Importamos las librerías necesarias para construir y entrenar la red.
# NumPy nos ayuda a trabajar con arreglos numéricos de forma eficiente.
# re permite limpiar texto, quitando caracteres que no aportan significado.
# Tokenizer convierte palabras en números para que la red pueda procesarlas.
# pad_sequences ajusta todas las frases a la misma longitud, algo obligatorio en redes neuronales.
# Sequential nos permite crear el modelo capa por capa.
# Embedding transforma palabras en vectores densos, LSTM aprende secuencias y Dense produce la salida final.
# EarlyStopping detiene el entrenamiento si el modelo deja de mejorar, evitando sobreajuste.
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

## 1. Datos de entrenamiento

Creamos un conjunto pequeño de frases y sus etiquetas: 1 para positivas y 0 para negativas.

In [30]:
# En esta lista guardamos las frases que usaremos para entrenar la red.
# Cada frase representa un ejemplo de texto que debe ser clasificado como positivo o negativo.
# La variable train_labels contiene la etiqueta correcta de cada frase:
# 1 significa sentimiento positivo y 0 significa sentimiento negativo.
# Cuanto más variado sea este conjunto, mejor podrá aprender la red a generalizar.
train_texts = [
    "me encanta este producto",
    "me gusta mucho",
    "excelente servicio",
    "estoy muy feliz",
    "me siento genial",
    "es una gran experiencia",
    "super recomendable",
    "lo mejor que he comprado",
    "muy contento con la compra",
    "adoro este lugar",
    "me encanta su calidad",
    "me pareció maravilloso",
    "simplemente excelente",
    "estoy muy satisfecho",
    "una compra fantástica",
    "lo odio",
    "es terrible",
    "no me gustó nada",
    "muy malo",
    "es una experiencia horrible",
    "pesimo servicio",
    "me decepcionó totalmente",
    "horrible",
    "no lo recomiendo",
    "me arrepiento de comprarlo",
    "fue una mala decisión",
    "definitivamente no lo volvería a comprar",
    "estoy muy disgustado",
    "una experiencia pésima",
    "está muy mal hecho",
    "me irrita mucho",
]
train_labels = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

## 2. Preparación del texto

Convertimos las palabras en números para que la red pueda procesarlas.

In [31]:
def limpiar_texto(texto):
    # Convertimos todo a minúsculas para que palabras como "Excelente" y "excelente" se traten igual.
    texto = texto.lower()
    # Eliminamos signos de puntuación, tildes y otros caracteres que no aportan información semántica.
    # Esto ayuda a que el modelo se centre en las palabras reales del texto.
    texto = re.sub(r"[^a-záéíóúñ\s]", "", texto)
    return texto

# Aplicamos la limpieza a todas las frases de entrenamiento.
# Esto produce una versión más uniforme del texto, lo cual mejora la capacidad de aprendizaje del modelo.
train_texts_limpios = [limpiar_texto(texto) for texto in train_texts]

# Tokenizer convierte palabras en índices numéricos.
# Cada palabra única recibe un número, de modo que la red pueda trabajar con datos numéricos.
# El parámetro oov_token permite manejar palabras nuevas que no aparecieron en el entrenamiento.
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts_limpios)

# Transformamos cada frase en una secuencia de números según el vocabulario aprendido.
sequences = tokenizer.texts_to_sequences(train_texts_limpios)

# Como las frases pueden tener longitudes diferentes, debemos ajustarlas.
# pad_sequences añade ceros al final de las frases más cortas para que todas tengan la misma longitud.
# Esto es necesario porque las capas neuronales esperan entradas de tamaño fijo.
max_len = 10
X = pad_sequences(sequences, maxlen=max_len, padding="post", truncating="post")
y = np.array(train_labels)

## 3. Construcción del modelo

Usamos una capa de embeddings y una LSTM para aprender patrones en las frases.

In [32]:
# Creamos el modelo secuencial, es decir, una estructura en la que las capas se apilan una tras otra.
# La primera capa, Embedding, convierte cada palabra en un vector de números que representa su significado de forma densa.
# SpatialDropout1D ayuda a reducir el sobreajuste, evitando que la red memorice demasiado los datos de entrenamiento.
# LSTM es la capa recurrente que aprende dependencias entre palabras a lo largo de la frase.
# Dense transforma la salida de la LSTM en una decisión final: positiva o negativa.
# La última capa usa sigmoid porque la salida debe ser un valor entre 0 y 1, interpretado como probabilidad.
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=32, input_length=max_len),
    SpatialDropout1D(0.2),
    LSTM(64, return_sequences=False),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Compilamos el modelo indicando cómo se va a entrenar.
# Adam es un optimizador eficiente que ajusta los pesos de la red automáticamente.
# binary_crossentropy es la pérdida adecuada para problemas de clasificación binaria como este.
# accuracy permite ver qué tan bien está clasificando el modelo durante el entrenamiento.
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

## 4. Entrenamiento

Entrenamos la red con los ejemplos disponibles.

In [33]:
# Entrenamos la red con los ejemplos preparados previamente.
# El entrenamiento consiste en ajustar los pesos de la red para que pueda asociar ciertos patrones de palabras con etiquetas positivas o negativas.
# EarlyStopping supervisa la pérdida del modelo y detiene el proceso si ya no mejora durante varias épocas.
# Esto evita entrenar demasiado y reducir la capacidad de generalización del modelo.
callbacks = [EarlyStopping(monitor="loss", patience=3, restore_best_weights=True)]
history = model.fit(X, y, epochs=50, batch_size=2, verbose=0, callbacks=callbacks)

loss = history.history.get("loss", [])
accuracy = history.history.get("accuracy", [])

print("Métricas de entrenamiento:")
print(f"- Pérdida final: {loss[-1]:.4f}")
print(f"- Precisión final: {accuracy[-1]:.4f}")

Métricas de entrenamiento:
- Pérdida final: 0.0001
- Precisión final: 1.0000


## 5. Predicción

Probamos el modelo con nuevas frases para ver su comportamiento.

In [34]:
def predecir_sentimiento(texto):
    # Antes de clasificar una frase nueva, la preprocesamos igual que lo hicimos con los datos de entrenamiento.
    # Esto es crucial porque la red solo entiende texto transformado a números.
    texto_limpio = limpiar_texto(texto)
    # Convertimos la frase a una secuencia numérica utilizando el vocabulario aprendido durante el entrenamiento.
    secuencia = tokenizer.texts_to_sequences([texto_limpio])
    # Ajustamos la longitud de la secuencia para que coincida con las entradas esperadas por la red.
    secuencia = pad_sequences(secuencia, maxlen=max_len, padding="post", truncating="post")
    # El modelo devuelve una probabilidad entre 0 y 1.
    # Valores cercanos a 1 indican que la red cree que la frase es positiva.
    probabilidad = model.predict(secuencia, verbose=0)[0][0]

    # Si la probabilidad es mayor o igual a 0.5, consideramos la frase como positiva.
    # En caso contrario, la clasificamos como negativa.
    if probabilidad >= 0.5:
        etiqueta = "Positivo"
    else:
        etiqueta = "Negativo"

    return etiqueta, probabilidad

# Mostramos una pequeña tabla de resultados con la etiqueta y la probabilidad estimada.
resultados = []
for texto in [
    "me encanta",
    "me encanta mucho",
    "no me gustó para nada",
    "es un desastre",
    "increíble experiencia",
    "muy buena calidad",
    "horrible y decepcionante",
    "excelente",
]:
    etiqueta, probabilidad = predecir_sentimiento(texto)
    resultados.append((texto, etiqueta, round(float(probabilidad), 4)))

# Métricas de entrenamiento del modelo
loss = model.history.history.get("loss", [])
accuracy = model.history.history.get("accuracy", [])

print("Métricas del entrenamiento:")
print(f"- Pérdida final: {loss[-1]:.4f}")
print(f"- Precisión final: {accuracy[-1]:.4f}")
print("\nPredicciones:")
for texto, etiqueta, prob in resultados:
    print(f"{texto} -> {etiqueta} ({prob})")

Métricas del entrenamiento:
- Pérdida final: 0.0001
- Precisión final: 1.0000

Predicciones:
me encanta -> Positivo (0.9998)
me encanta mucho -> Positivo (0.9998)
no me gustó para nada -> Negativo (0.0)
es un desastre -> Positivo (0.7296)
increíble experiencia -> Negativo (0.0009)
muy buena calidad -> Positivo (0.8503)
horrible y decepcionante -> Negativo (0.0)
excelente -> Positivo (0.9998)
